# Bonus: Racetrack Environment

We extend our RL project to the **racetrack-v0** environment from highway-env.
This is a fundamentally different task: the agent must **follow a curved track**
using **continuous steering** while avoiding a single other vehicle.

Key differences from highway:
- **Continuous actions** (steering angle) vs discrete meta-actions
- **Curved track** vs straight highway
- **300-step episodes** vs 40 steps
- **OccupancyGrid / Kinematics** observations

We apply two algorithms:
1. **PPO (SB3)** with native continuous actions
2. **DQN (manual, PyTorch)** with discretized steering (7 bins)

All lessons from the highway project are applied: lr schedules, gradient
clipping, eval-based model selection, no early stopping.

## Setup

In [ ]:
import os
import numpy as np
import gymnasium as gym
import highway_env  # noqa: F401
import torch
import torch.nn as nn
import torch.optim as optim
import random
import json
import matplotlib.pyplot as plt
from pathlib import Path
from collections import deque
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from tqdm import tqdm

ENV_ID = "racetrack-v0"
RETRAIN = False

RACETRACK_CONFIG_PPO = {
    "observation": {
        "type": "OccupancyGrid",
        "features": ["presence", "on_road"],
        "grid_size": [[-18, 18], [-18, 18]],
        "grid_step": [3, 3],
        "as_image": False,
        "align_to_vehicle_axes": True,
    },
    "action": {"type": "ContinuousAction", "longitudinal": False, "lateral": True},
    "duration": 300,
    "simulation_frequency": 15,
    "policy_frequency": 5,
    "other_vehicles": 1,
}

RACETRACK_CONFIG_DQN = {
    "observation": {
        "type": "Kinematics",
        "vehicles_count": 5,
        "features": ["presence", "x", "y", "vx", "vy"],
        "normalize": True,
    },
    "action": {"type": "ContinuousAction", "longitudinal": False, "lateral": True},
    "duration": 300,
    "simulation_frequency": 15,
    "policy_frequency": 5,
    "other_vehicles": 1,
}

STEERING_BINS = np.array([-1.0, -0.67, -0.33, 0.0, 0.33, 0.67, 1.0])

Path("models").mkdir(exist_ok=True)
Path("figures").mkdir(exist_ok=True)
Path("videos").mkdir(exist_ok=True)

print(f"Racetrack bonus ready | RETRAIN={RETRAIN}")

## Environment Exploration

In [ ]:
env = gym.make(ENV_ID, config=RACETRACK_CONFIG_PPO)
obs, info = env.reset()

print(f"Observation space: {env.observation_space}")
print(f"Observation shape: {obs.shape}")
print(f"Action space: {env.action_space}")
print(f"Action space low/high: {env.action_space.low} / {env.action_space.high}")
print(f"\nThis is CONTINUOUS lateral steering in [-1, 1].")
print(f"Duration: 300 steps (~60s at 5 Hz policy frequency)")
print(f"\nOccupancyGrid observation (12x12x2 = 288 features):")
print(f"  Shape: {obs.shape}")
env.close()

## Utility functions

In [ ]:
def evaluate_racetrack(model, config, num_episodes=20):
    env = make_vec_env(ENV_ID, n_envs=4, env_kwargs={"config": config})
    ep_rewards, ep_lengths = [], []
    current_rewards = np.zeros(4)
    current_lengths = np.zeros(4)
    obs = env.reset()
    pbar = tqdm(total=num_episodes, desc="Evaluating")
    while len(ep_rewards) < num_episodes:
        action, _ = model.predict(obs, deterministic=True)
        obs, rewards, dones, infos = env.step(action)
        current_rewards += rewards
        current_lengths += 1
        for i, done in enumerate(dones):
            if done:
                ep_rewards.append(current_rewards[i])
                ep_lengths.append(current_lengths[i])
                current_rewards[i] = 0
                current_lengths[i] = 0
                pbar.update(1)
                if len(ep_rewards) >= num_episodes:
                    break
    pbar.close()
    env.close()
    mean_r = np.mean(ep_rewards)
    std_r = np.std(ep_rewards)
    mean_l = np.mean(ep_lengths)
    print(f"  Reward: {mean_r:.2f} +/- {std_r:.2f} | Length: {mean_l:.0f}")
    return mean_r, mean_l

## Algorithm 1: PPO (SB3) — Continuous steering

PPO is the natural choice for continuous action spaces. Unlike DQN which
outputs Q-values for discrete actions, PPO's actor network outputs the
**mean and standard deviation** of a Gaussian distribution over the
steering angle. Actions are sampled from this distribution during training
and taken at the mean during evaluation.

Compared to our highway PPO:
- `gamma=0.99` (vs 0.9): episodes are 300 steps, requiring a longer planning horizon
- `n_steps=1024` (vs 512): larger batches for the longer episodes
- `300k timesteps` (vs 80k): proportional to the longer episodes
- Same lr schedule (3e-4 -> 3e-5) and entropy bonus (0.01)

In [ ]:
if RETRAIN or not Path("models/racetrack_ppo.zip").exists():
    print("Training PPO for racetrack... (run train_racetrack.py first)")
    print("Set RETRAIN=True or run: uv run python bonus/train_racetrack.py")
else:
    print("Loading pre-trained PPO model...")
    model_ppo = PPO.load("models/racetrack_ppo")

histories = {}

In [ ]:
if Path("models/racetrack_ppo.zip").exists():
    print("=== PPO Evaluation ===")
    ppo_r, ppo_l = evaluate_racetrack(model_ppo, RACETRACK_CONFIG_PPO)
    histories["PPO (continuous)"] = {"reward": ppo_r, "length": ppo_l}
else:
    print("PPO model not found. Run train_racetrack.py first.")

## Algorithm 2: DQN Manual — Discretized steering

Our manual DQN from the highway project cannot handle continuous actions
natively. Instead of switching to a different algorithm, we **discretize**
the steering space into 7 bins: `[-1.0, -0.67, -0.33, 0.0, 0.33, 0.67, 1.0]`.

The Q-network outputs 7 Q-values (one per bin), the argmax selects the bin
index, and we map it back to the continuous steering value. This shows how
a discrete algorithm can be adapted to continuous problems.

We use Kinematics observations (5x5 = 25 features) instead of OccupancyGrid
to keep the same input format as our highway DQN.

Compared to our highway DQN manual:
- `gamma=0.95` (vs 0.8): longer episodes need longer horizon
- `buffer_size=30000` (vs 15000): more data to store
- `epsilon_decay_steps=10000` (vs 5000): proportional to budget
- `max_grad_norm=10.0`: gradient clipping (Phase 4 lesson)
- 2000 episodes with periodic deterministic eval (Phase 4 lesson)

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, obs_size, n_actions, hidden_sizes=(256, 256)):
        super().__init__()
        layers = []
        prev = obs_size
        for h in hidden_sizes:
            layers.extend([nn.Linear(prev, h), nn.ReLU()])
            prev = h
        layers.append(nn.Linear(prev, n_actions))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action_idx, reward, next_state, done):
        self.buffer.append((state, action_idx, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            np.array(states, dtype=np.float32),
            np.array(actions, dtype=np.int64),
            np.array(rewards, dtype=np.float32),
            np.array(next_states, dtype=np.float32),
            np.array(dones, dtype=np.float32),
        )

    def __len__(self):
        return len(self.buffer)


class RacetrackDQNAgent:
    """DQN with discretized continuous actions for racetrack."""

    def __init__(self, obs_size, steering_bins, hidden_sizes=(256, 256),
                 lr=5e-4, gamma=0.95, buffer_size=30000, batch_size=64,
                 target_update_freq=100, epsilon_start=1.0, epsilon_end=0.05,
                 epsilon_decay_steps=10000, max_grad_norm=10.0, device="cpu"):
        self.device = torch.device(device)
        self.steering_bins = steering_bins
        self.n_actions = len(steering_bins)
        self.gamma = gamma
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq
        self.max_grad_norm = max_grad_norm
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = (epsilon_start - epsilon_end) / epsilon_decay_steps
        self.q_net = QNetwork(obs_size, self.n_actions, hidden_sizes).to(self.device)
        self.target_net = QNetwork(obs_size, self.n_actions, hidden_sizes).to(self.device)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.buffer = ReplayBuffer(buffer_size)
        self.step_count = 0
        self.training_rewards = []

    def predict(self, obs, deterministic=False):
        obs_np = np.array(obs, dtype=np.float32)
        if obs_np.ndim == 3:
            obs_flat = obs_np.reshape(obs_np.shape[0], -1)
        elif obs_np.ndim == 2:
            obs_flat = obs_np.reshape(1, -1)
        else:
            obs_flat = obs_np.reshape(1, -1)
        with torch.no_grad():
            t = torch.FloatTensor(obs_flat).to(self.device)
            indices = self.q_net(t).argmax(dim=1).cpu().numpy()
        actions = np.array([[self.steering_bins[i]] for i in indices])
        if obs_np.ndim == 2 and obs_np.shape[0] != 1:
            return actions[0], None
        return actions, None

    def save(self, path):
        torch.save({
            "q_net": self.q_net.state_dict(),
            "target_net": self.target_net.state_dict(),
            "epsilon": self.epsilon,
            "step_count": self.step_count,
            "training_rewards": self.training_rewards,
        }, path)

    def load(self, path):
        ckpt = torch.load(path, map_location=self.device, weights_only=False)
        self.q_net.load_state_dict(ckpt["q_net"])
        self.target_net.load_state_dict(ckpt["target_net"])
        self.epsilon = ckpt.get("epsilon", self.epsilon_end)
        self.step_count = ckpt.get("step_count", 0)
        self.training_rewards = ckpt.get("training_rewards", [])

In [ ]:
if torch.backends.mps.is_available():
    DEVICE_MANUAL = "mps"
elif torch.cuda.is_available():
    DEVICE_MANUAL = "cuda"
else:
    DEVICE_MANUAL = "cpu"

env_tmp = gym.make(ENV_ID, config=RACETRACK_CONFIG_DQN)
obs_shape = env_tmp.observation_space.shape
obs_size = int(np.prod(obs_shape))
env_tmp.close()

agent = RacetrackDQNAgent(
    obs_size=obs_size, steering_bins=STEERING_BINS,
    hidden_sizes=(256, 256), lr=5e-4, gamma=0.95,
    buffer_size=30000, batch_size=64, target_update_freq=100,
    epsilon_start=1.0, epsilon_end=0.05, epsilon_decay_steps=10000,
    max_grad_norm=10.0, device=DEVICE_MANUAL,
)

if not RETRAIN and Path("models/racetrack_dqn_manual.pt").exists():
    print("Loading pre-trained DQN manual model...")
    agent.load("models/racetrack_dqn_manual.pt")
else:
    print("DQN model not found. Run train_racetrack.py first.")
    print("Set RETRAIN=True or run: uv run python bonus/train_racetrack.py")

In [ ]:
if Path("models/racetrack_dqn_manual.pt").exists():
    print("=== DQN Manual Evaluation ===")
    dqn_r, dqn_l = evaluate_racetrack(agent, RACETRACK_CONFIG_DQN)
    histories["DQN (discretized)"] = {"reward": dqn_r, "length": dqn_l}
else:
    print("DQN model not found.")

## Benchmark

In [ ]:
if histories:
    print("=" * 50)
    print("RACETRACK BENCHMARK")
    print("=" * 50)
    print(f"{'Algorithm':<25} {'Reward':>10} {'Length':>10}")
    print("-" * 50)
    for name, data in histories.items():
        print(f"{name:<25} {data['reward']:>10.2f} {data['length']:>10.0f}")
    print("=" * 50)

In [ ]:
if histories:
    fig, ax = plt.subplots(figsize=(8, 5))
    names = list(histories.keys())
    rewards = [histories[n]["reward"] for n in names]
    colors = ["#4CAF50", "#FF9800"]
    bars = ax.bar(names, rewards, color=colors[:len(names)], edgecolor="black", linewidth=0.5)
    ax.set_ylabel("Mean Reward (20 episodes)")
    ax.set_title("Racetrack: PPO (continuous) vs DQN (discretized)")
    ax.grid(axis="y", alpha=0.3)
    for bar, val in zip(bars, rewards):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{val:.2f}", ha="center", va="bottom", fontweight="bold")
    plt.tight_layout()
    plt.savefig("figures/racetrack_benchmark.png", dpi=150)
    plt.show()
    print("Saved to figures/racetrack_benchmark.png")

In [ ]:
if hasattr(agent, "training_rewards") and len(agent.training_rewards) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    rewards = agent.training_rewards
    window = 50
    smoothed = [np.mean(rewards[max(0,i-window):i+1]) for i in range(len(rewards))]
    ax.plot(rewards, alpha=0.2, label="Episode reward")
    ax.plot(smoothed, linewidth=2, label=f"Moving avg (window={window})")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Reward")
    ax.set_title("Racetrack DQN (discretized) — Training Curve")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("figures/racetrack_dqn_training.png", dpi=150)
    plt.show()
    print("Saved to figures/racetrack_dqn_training.png")

## Analysis: Highway vs Racetrack

### Key differences

| Aspect | Highway | Racetrack |
|--------|---------|----------|
| Actions | 5 discrete meta-actions | Continuous steering [-1, 1] |
| Observation | Kinematics 5x5 | OccupancyGrid 12x12x2 |
| Duration | 40 steps | 300 steps |
| Challenge | Dense traffic, straight road | Track following, curves |
| Best gamma (highway) | 0.8-0.9 | 0.95-0.99 (longer horizon) |

### Transfer of lessons learned

The iterative methodology from our highway project transferred well:
- **Lr schedules** prevent late-training instability in both environments
- **Gradient clipping** stabilizes the manual DQN regardless of the task
- **Eval-based model selection** is essential when training reward is noisy
- **No early stopping** with longer patience lets the agent fully exploit

### Continuous vs discrete actions

PPO handles continuous actions natively (Gaussian policy), making it the
natural choice for racetrack. Our discretized DQN approach (7 steering bins)
shows that a discrete algorithm **can** be adapted to continuous problems,
but with a loss of precision: 7 bins cannot represent fine steering adjustments
needed in tight curves.

### Limitations

- Discretization granularity (7 bins) may be too coarse for optimal performance
- Single run per algorithm (no multi-seed averaging)
- OccupancyGrid vs Kinematics makes direct PPO/DQN comparison imperfect
  (different observation spaces)